# 非線形バネモデルへの入力制限を行うC/GMRESの適用

[非線形バネモデルへのC/GMRES](nonlinear_spring.ipynb)では通常の状態方程式の制約のみであった。

このNoteでは制御入力 $u \le u_{max}$ の制約を行うC/GMRESの適用を実施する。

## モデリング

非線形バネモデルは[通常のC/GMRES](nonlinear_spring.ipynb)で用いた1質点モデルとする。

<img src="images/one-degree-nonlinear-spring.png" style="width:40%;">

状態$X(t)$を以下のようにすると、

$$
X(t) = [x(t), v(t)]^T
$$

状態方程式 $\dot{X} = f(X, u)$ は次のようになる。

$$
\frac{d}{dt}\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix} =
\begin{bmatrix}
v(t) \\ \dfrac{1}{m}\left(u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
$$

### 評価関数とコスト

制御入力 $u$ の制約はダミー入力 $u_d$ を用いて次の等式制約$C(u, u_d)$となる。

$$
C(u, u_d) = u(t)^2 + u_d(t)^2 - u^2_{max} = 0
$$

$C$を用いて拡大評価関数$\bar{J}$は次のように構成される。

$$
\bar{J} = \Phi(X(T)) + \int_0^T \left[ L_d(X, u, u_d) + \lambda^T \left(f(X, u) - \dot{X} \right) + \rho C(u, u_d) \right] dt
$$

#### ランニングコスト $L_d$

ランニングコスト $L_d(X, u, u_d)$は次のように設定する。入力制限のため $r_d u_d, r_d > 0$ の項目を設定している。

$$
L_d(X, u, u_d) = \frac{1}{2}q_x(x(t) - x_{ref})^2 + \frac{1}{2}q_v v(t)^2 + \frac{1}{2} r u(t)^2 - r_d u_d
$$

$\bar{J}$の最小化問題であるため、ゲイン $q_x, q_v, r$に関する項は次の意味となる。

- $q_x$ : 位置の目標値と誤差を小さくする
- $q_v$ : 速度を大きくしない
- $r$ : 入力を大きくしない

また、 $-r_d u_d$の設定により、$r_d >0$ となるように選択する。

#### 終端コスト $\Phi$

終端コスト $\Phi(X(T))$ は次のように設定する。

$$
\Phi(X(T)) = \frac{1}{2}q_{xT}(x(T) - x_{ref})^2 + \frac{1}{2} q_{vT} v(T)^2
$$

$\bar{J}$の最小化問題であるため、ゲイン $q_{xT}, q_{vT}$に関する項は次の意味となる。

- $q_{xT}$ : 時刻$T$における位置の目標値と誤差を小さくする
- $q_{vT}$ : 時刻$T$における速度を大きくしない

#### 随伴変数　$\lambda$

$\lambda$は状態方程式制約$f(X,u) - \dot{X}$ と内積をとり、$\lambda^T (f(X,u) - \dot{X}) $がスカラーとなるように設定する。状態$X$は2変数であるため、$\lambda$を次のように設定する。

$$
\lambda = [\lambda_x, \lambda_v]^T
$$

#### 等式制約の係数 $\rho$

$C(u, u_d)$がスカラーであるため、$\rho$もスカラーである。


### PMP 条件

Hamiltonin $H$ は次の式である。

$$
H(X, u, u_d, \lambda) = L_d (X, u, u_d) + \lambda^T f(X,u) + \rho C(u, u_d)
$$

$$
H = \frac{1}{2} q_x(x(t) - x_{ref})^2 + \frac{1}{2} q_v v(t)^2 + \frac{1}{2} r u(t)^2 - r_d u_d + \lambda_x v(t) + \frac{\lambda_v}{m}\left(u(t) - k x(t) - k_3 x(t)^3  \right) + \rho (u^2(t) + u_d^2(t) - u^2_{max})
$$

PMP条件を構成するため、$H$ を$X,u, u_d$で偏微分、$\Phi$を$X(T)$ で偏微分を行う。また、$H_\lambda = f(X,u)$ である。

$$
\begin{aligned}
H_X &= \frac{\partial H}{\partial X} = \begin{bmatrix} \partial H / \partial x \\ \partial H / \partial v \end{bmatrix} =
\begin{bmatrix}
q_x(x(t) - x_{ref}) - \lambda_v/m \ \left( k + 3 k_3 x(t)^2 \right) \\
q_v v(t) + \lambda_x 
\end{bmatrix} \\
H_u &= \frac{\partial H}{\partial u} = r u(t) + \frac{\lambda_v}{m} + 2\rho u(t)\\
H_{u_d} &= \frac{\partial H}{\partial u_d} = -r_d + 2\rho u_d(t)\\
\Phi_X &= \frac{\partial \Phi}{\partial X} = \begin{bmatrix} \partial \Phi / \partial x \\ \partial \Phi / \partial v \end{bmatrix} =
\begin{bmatrix}
q_{xT}(x(T) - x_{ref}) \\
q_{vT} v(T)
\end{bmatrix} 
\end{aligned}
$$



$H$を用いて、PMP条件は次のようになる。

$$
\begin{array}{l}
\dot{X} = H_\lambda \\
\lambda(T) = \Phi_X(X(T)) \\
\dot{\lambda} = - H_X \\
H_u = 0 \\
H_{u_d} = 0 \\
\end{array}
$$ 

これに加え、$C(u, u_d)=0$ も守るべき条件となる。

よって1質点の非線形バネモデルにおけるPMP条件の式は以下となる。

状態方程式

$$
\boxed{
\frac{d}{dt}
\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix}
= \begin{bmatrix}
v(t) \\ \dfrac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
}
$$

終端条件

$$
\boxed{
\begin{bmatrix}
\lambda_x(T) \\ \lambda_v(T)
\end{bmatrix} =
\begin{bmatrix}
q_{xT}(x(T) - x_{ref}) \\
q_{vT} v(T)
\end{bmatrix} 
}
$$

随伴方程式

$$
\boxed{
\frac{d}{dt}
\begin{bmatrix}
\lambda_x \\ \lambda_v
\end{bmatrix}
=-\begin{bmatrix}
q_x(x(t) - x_{ref}) - \lambda_v/m \ \left( k + 3 k_3 x(t)^2 \right) \\
q_v v(t) + \lambda_x 
\end{bmatrix}
}
$$

停留条件

$$
\boxed{
F=
\begin{bmatrix}
r u(t) + \frac{\lambda_v}{m}  + 2\rho u(t) \\
-r_d + 2\rho u_d(t)  \\
u(t)^2 + u_d(t)^2 - u^2_{max} 
\end{bmatrix} = 0
}
$$



#### 求めるべき $U$ の構成

C/GMRESでは予測ホライゾン上の各時刻$\tau$で、上記の式を満たす必要がある。上式では、$u(\tau),u_d(\tau), \rho(\tau)$が未定の値である。よって、$U(t_k)$ を次のように設定する。

$$
U(t_k) = [u_0(t_k), u_{d_0}(t_k) , \rho_0(t_k) , \cdots , u_{N-1}(t_k), u_{d_{N-1}}(t_k) , \rho_{N-1}(t_k)]^T
$$

### 状態 $X$ 随伴変数 $\lambda$ の時系列の計算式と停留条件式 $F$ の構成

入力列 $U(t_k)$ 、状態の初期値 $X[0]$、予測ホライゾンの長さ$T(t_k)=T_f(1 - \exp(-\alpha t_k))$ の$t_k$を設定値として時系列を計算する。

予測ホライゾンのステップ幅 $h(t_k)$ は $T(t_k)$の分割数 $N$ より以下のようになる。

$$
h(t_k) = T(t_k) / N
$$

以下の式は上から順に、状態$X[n]$、終端状態 $\lambda[N]$、随伴変数 $\lambda[n]$ に関する。

$$
\begin{aligned}
\begin{bmatrix}
x[n+1] \\ v[n+1]
\end{bmatrix} &=
\begin{bmatrix}
x[n] \\ v[n]
\end{bmatrix} +
h\begin{bmatrix}
v[n] \\ \dfrac{1}{m}\left( u_n(t_k) - k x[n] - k_3 x[n]^3 \right)
\end{bmatrix}, \quad n = 0, \cdots , N-1 \\
\begin{bmatrix}
\lambda_x[N] \\ \lambda_v[N]
\end{bmatrix} &=
\begin{bmatrix}
q_{xT}(x[N] - x_{ref}) \\
q_{vT} v[N]
\end{bmatrix} \\
\begin{bmatrix}
\lambda_x[n] \\ \lambda_v[n]
\end{bmatrix} &=\begin{bmatrix}
\lambda_x[n+1] \\ \lambda_v[n+1]
\end{bmatrix}
+h\begin{bmatrix}
q_x(x[n] - x_{ref}) - \lambda_v[n+1]/m \ \left( k + 3 k_3 x[n]^2 \right) \\
q_v v[n] + \lambda_x[n+1] 
\end{bmatrix} , \quad n = N-1, \cdots , 1
\end{aligned}
$$

上記を用いて、停留条件式 $F(X(t_k), U(t_k))$ を以下のように構成する。

$$
F(X(t_k), U(t_k)) = 
\begin{bmatrix}
r u_0(t_k) + \dfrac{\lambda_v[1]}{m} + 2\rho_0(t_k) u_0(t_k)\\
-r_d + 2\rho_0(t_k) u_{d_0}(t_k)  \\
u_0(t_k)^2 + u_{d_0}(t_k)^2 - u^2_{max} \\
\vdots \\
r u_{N-1}(t_k) + \dfrac{\lambda_v[N]}{m} + 2\rho_{N-1}(t_k) u_{N-1}(t_k)\\
-r_d + 2\rho_{N-1}(t_k) u_{d_{N-1}}(t_k)  \\
u_{N-1}(t_k)^2 + u_{d_{N-1}}(t_k)^2 - u^2_{max} \\
\end{bmatrix} = 0
$$

また、この時系列から$F$を構成する計算はC/GMRESの実行中に何度も行われる。そのため

$$
F(U(t_k), X(t_k), t_k)
$$

を関数として定義する。これは、$U(t_k)$ に制御入力列、$X(t_k)$ に実時間 $t_k$ における現在状態を与え、それを予測区間の初期状態$X[0]$として使用する、$t_k$を$T(t_k)$ を計算する入力として与えれれば、それをもとに停留条件の式 $F(U(t_k), X(t_k), t_k)$ を構成する、という意味である。

#### 初期 $U(0)$ の計算

求めるべき変数が $u(0), u_d(0), \rho(0)$ と3つであるため、Newton-Raphson法により求める。

以下の$F(U)$を $U = [u, u_d, \rho]$ で偏微分してヤコビアン$J_F$を求める。

$$
F(U)= \begin{bmatrix}
H_u \\ H_{u_d} \\ C 
\end{bmatrix} = 
\begin{bmatrix}
r u(t) + \frac{\lambda_v}{m}  + 2\rho u(t) \\
-r_d + 2\rho u_d(t)  \\
u(t)^2 + u_d(t)^2 - u^2_{max} 
\end{bmatrix}
$$

$$
J_F(U) = \frac{\partial F}{\partial U} =
\begin{bmatrix}
\dfrac{\partial H_u}{\partial u} & \dfrac{\partial H_u}{\partial u_d} & \dfrac{\partial H_u}{\partial \rho} \\ 
\dfrac{\partial H_{u_d}}{\partial u} & \dfrac{\partial H_{u_d}}{\partial u_d} & \dfrac{\partial H_{u_d}}{\partial \rho} \\ 
\dfrac{\partial C}{\partial u} & \dfrac{\partial C}{\partial u_d} & \dfrac{\partial C}{\partial \rho} \\ 
\end{bmatrix} =
\begin{bmatrix}
r + 2 \rho & 0 & 2 u(t) \\
0 & 2 \rho & 2 u_d(t) \\
2 u(t) & 2 u_d(t) & 0
\end{bmatrix}
$$


初期 $U^{(0)}=[u^{(0)}, u_d^{(0)}, \rho^{(0)}] $ を決め、$F(U),J_F(U)$を用いて、以下の式をLU分解で解き、$\Delta U$ を求る。

$$
J_F(U^{(j)}) \Delta U = -F(U^{(j)})
$$

次に以下のように$U^{(j+1)}$ を更新する。

$$
U^{(j+1)} = U^{(j)} + \Delta U
$$

そして、$||F(U^{(j+1)})||$が以下のように十分小さくなるまで上記を繰り返す。

$$
||F(U^{j+1})|| < \sigma
$$

ループが停止したときの$U^{(j+1)} = [u^{(j+1)}, u_d^{(j+1)}, \rho^{(k+1)}]$を$U(0)$の要素$[u(0), u_d(0), \rho(0)]$と決定し、以下のように$U(0)$を予測ホライゾンの区間の離散時間に対応させた形で構成する。

$$
U(0) = [u(0), u_d(0), \rho(0), \cdots , u(0), u_d(0), \rho(0)]^T \in \mathbb{R}^{(3 \times N)}
$$



以上がC/GMRESを計算するために必要設定である。

これらの計算を用いてC/GMRESの制御ループを以下のように実行する。

## C/GMRESの計算

### Step 1 状態取得

制御対象の状態 $X(t_k)$ を取得する。

### Step 2 制御入力列の更新

入力列 $U(t_k) \ \in \mathbb{R}^{(3\times N)}$ は前回のC/GMRESで計算された値を用いる。

$t_0$ の場合、上記のNewton-Raphson法で $u(0), u_d(0), \rho(0)$ を計算し、$U(t_0) \ \in \mathbb{R}^{(3\times N)}$ の入力列を構成する。

$$
U(t_0) = [u(0), u_d(0), \rho(0), \cdots , u(0), u_d(0), \rho(0)]^T
$$

### Step 3 制御対象へ制御入力を出力

制御対象へ 制御入力列 $U(t_k)$ の第一要素 $u_0(t_k)$を出力する。

### Step 4 状態$\dot{X}$の計算

$\dot{X}(t_k) = f(X(t_k), u_0(t_k))$ を計算する。

今回の場合、以下の状態方程式を計算する。

$$
\frac{d}{dt}
\begin{bmatrix}
x(t_k) \\ v(t_k)
\end{bmatrix}
= \begin{bmatrix}
v(t_k) \\ \dfrac{1}{m}\left( u_0(t_k) - k x(t_k) - k_3 x(t_k)^3 \right)
\end{bmatrix}
$$

### Step 5 停留条件の計算

$F(U(t_k), X(t_k), t_k)$ を計算する。

### Step 6 右辺 $b_k$ の計算

GMRESの右辺に渡す $b_k$ を計算する。

$F(U(t_k) , X(t_k) + \varepsilon \dot{X}(t_k) , t_k + \varepsilon)$ を計算する。

$$
b_k = - \frac{F(U(t_k) , X(t_k) + \varepsilon \dot{X}(t_k) , t_k + \varepsilon) - F(U(t_k), X(t_k), t_k)}{\varepsilon} - \zeta F(U(t_k), X(t_k), t_k)
$$

### Step 7 GMRESの計算

#### Step 7-1 初期残差の計算

初期残差 $r_0$ の計算に用いる初期解を $\dot{U}^{(0)}(t_k)$を前回の制御ループで計算した $\dot{U}(t_{k-1})$ とする。

$t_0$ の場合、$\dot{U}^{(0)}(t_0)=0$ とする。

$t_k$ の場合、$r_0$ を計算する $F_U \dot{U}^{(0)}(t_k)$ を以下で計算する。

$$
F_U \dot{U}^{(0)}(t_k) = \frac{F(U(t_k) + \varepsilon \dot{U}^{(0)}(t_{k}), X(t_k), t_k) - F(U(t_k), X(t_k), t_k)}{ \varepsilon}
$$

以上より、初期残差を計算する。

$$
r_0 = b_k - F_U \dot{U}^{(0)}(t_k)
$$

#### Step 7-2 直交基底 $v_1$ の計算

$$
v_1 = r_ 0 / \beta , \quad \beta = ||r_0||
$$

#### Step 7-3 Arnoldi法 Step m (m=$1, \cdots $)

$$
\begin{aligned}
F_U(t_k) v_m &= \frac{F(U(t_k) + \varepsilon v_m, X(t_k), t_k) - F(U(t_k), X(t_k), t_k)}{\varepsilon} \\
h_{i, m} &= v_i^T F_U(t_k) v_m \quad (i = 1, \cdots , m) \\
w &= F_U(t_k) v_m - \sum_{i=1}^{m} h_{i,m} v_i \\
h_{m+1, m} &= ||w|| \\
v_{m+1} &= w / h_{m+1, m}
\end{aligned}
$$

この後、これまでのGivens回転を今回構成される 上ヘッセンベルグ行列の列へ適用し、上三角行列を作る。

残差 $g$ が目標よりも小さければ 最小二乗問題を解き、$Y_m$ を計算し、Arnoldi法の繰り返しを終了する。
そうでなければ、Arnoldi法のStep を繰り返す。

#### Step 7-4 $\dot{U}(t_k)$ の近似解を計算

Arnoldi法が停止した Step m では Krylov部分空間の直交基底 $V_m$ と、$Y_m$ が求まっている。
これを用いて、$\dot{U}(t_k)$ を次のように構成する。

$$
\dot{U}(t_k) = \dot{U}^{(0)}(t_k) + V_m Y_m
$$

### Step 8 次の制御入力列 $U(t_{k+1})$ を計算

求めた $\dot{U}(t_k)$ から、次の時刻の $U(t_{k+1})$ を以下の方法で計算する。

$$
U(t_{k+1}) = U(t_k) + \Delta t \  \dot{U}(t_k) , \quad \Delta t = t_{k} - t_{k-1}
$$
